# Full-pool judging — CLEF eHealth TAR 2017, abstract level

Supplies the auxiliary variable the model-assisted, Bayesian and stratified procedures need
and do not have: the judge's label on **every** pool document rather than on the
4.05 per cent covered by the frozen panel.

| | |
|---|---|
| Pairs to judge | **117,562** (30 topics) |
| Unique PubMed records to fetch | **99,304** |
| Arm | `qwen3-8b`, condition `C`, permissive guided decoding |
| Precision | bfloat16, no quantisation flag — fixed by `scale_prereg.py` |
| Device | one A100 40GB |
| Expected wall time | about 1.0 h at the 34.2 rows/s measured for this arm on the panel |

**Why this arm.** It has the highest specificity of the open-weight arms measured on the frozen
panel (0.132 false-positive rate), and 8B in bf16 leaves room for the KV cache at
`max_model_len` 4096. The choice was made from panel-measured rates and is disclosed as such;
the prediction registered in the plan concerns the whole grid of open arms, not this one.

**Resume.** Every stage is idempotent. Re-running a cell after a disconnect skips what is
already on disk: the fetch skips PMIDs already in `clef_abstracts.jsonl`, the judge skips
`(topic, pmid)` pairs already in its output. Shards are disjoint by a stable hash, so they can
run in separate sessions and be concatenated.

In [ ]:
# 1 · repository and dependencies
REPO = "https://github.com/Taekyoon/academic_ir_research_2026.git"

import os, subprocess, sys

# Resolve the checkout, THEN update it unconditionally. An earlier version of this cell
# returned early when the cwd was already the checkout, printing "already inside the
# checkout" and skipping the fetch - so re-running it after a fix was pushed left the old
# code on disk while reporting success. That is the same defect this cell's history already
# records once (a clone whose failure was swallowed by `|| echo`), and it cost a smoke run.
if os.path.basename(os.getcwd()) == "work" and os.path.isdir(".git"):
    print("already inside the checkout:", os.getcwd())
elif os.path.isdir("work/.git"):
    os.chdir("work"); print("reusing existing checkout:", os.getcwd())
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, "work"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("git clone failed (" + str(r.returncode) + "):\n" + r.stderr.strip())
    os.chdir("work"); print("cloned to:", os.getcwd())

# Always update. reset --hard touches TRACKED files only, so a fetched clef_abstracts.jsonl,
# the cached model weights and any labels/ output produced in this session all survive.
for cmd in (["git", "fetch", "--depth", "1", "origin", "main"],
            ["git", "reset", "--hard", "origin/main"]):
    rr = subprocess.run(cmd, capture_output=True, text=True)
    if rr.returncode != 0:
        raise SystemExit(" ".join(cmd) + " failed: " + rr.stderr.strip())

for p in ("code/pool_judge.py", "code/fetch_abstracts.py", "data/pool_pairs.csv",
          "data/pool_pmids.txt", "data/elig_prompt.txt", "data/crit_block_C.txt",
          "data/clef_topics.json", "prereg/scale_prereg.py", "prereg/baseline_prereg.py"):
    assert os.path.exists(p), "checkout is incomplete: " + p + " missing"
print("HEAD:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip())

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# BOTH versions are pinned, and transformers is the one that matters. vLLM 0.11.0 declares
# transformers>=4.55.2 with NO upper bound, so pip leaves Colab's preinstalled transformers
# 5.x in place - and transformers 5.x removed Tokenizer.all_special_tokens_extended, which
# vLLM 0.11.0 calls during tokenizer init. The run then dies with
#   AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended
# AFTER the model has already downloaded. Pinning transformers to the newest 4.x is the
# smallest fix; upgrading vLLM instead pulls a different pinned torch and a multi-gigabyte
# reinstall.
!pip -q install "vllm==0.11.0" "transformers==4.57.6" "huggingface_hub>=0.26"

import vllm, torch, transformers
print("vllm", vllm.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())
assert transformers.__version__.startswith("4."), (
    "transformers " + transformers.__version__ + " is a 5.x release and vLLM 0.11.0 cannot "
    "use its tokenizer API. Re-run this cell, then Runtime -> Restart session, then run it "
    "again.")


## 1 · Abstracts

PubMed text is fetched rather than redistributed. Resumable; supply your own contact address or omit the flag. About 500 batches of 200.

A fresh session fetches all 99,304. The 5,080 records already fetched for the frozen panel are not in the repository — abstract text is fetched rather than redistributed — so there is nothing to seed from, and the same extraction code produces the same text for those records.

In [ ]:
!python code/fetch_abstracts.py --pmids data/pool_pmids.txt \
    --out clef_abstracts.jsonl --sleep 0.4

In [ ]:
import json
have = {json.loads(l)['pmid'] for l in open('clef_abstracts.jsonl') if l.strip()}
want = {l.strip() for l in open('data/pool_pmids.txt') if l.strip()}
noabs = sum(1 for l in open('clef_abstracts.jsonl') if l.strip()
            and not json.loads(l)['abstract'].strip())
print(f'have {len(have):,} / {len(want):,} | absent {len(want - have):,} | no abstract body {noabs:,} ({noabs/max(len(have),1):.1%})')

## 2 · Smoke test

256 pairs. Check `strict_parse_rate` is 1.000 and `constraint_binding` is true **before** starting the full run. A non-binding result here means the guided target is not in force and the run must not proceed.

In [ ]:
!python code/pool_judge.py --arm qwen3-8b --condition C --guided \
    --pairs data/pool_pairs.csv --limit 256 --outdir labels_smoke

## 3 · Full run in shards

Four shards of about 29,390 pairs. Run them in order in one session, or one per session after a disconnect — the assignment is a stable hash, not a slice.

In [ ]:
import subprocess
for s in range(4):
    print(f'=== shard {s} ===', flush=True)
    subprocess.run(['python','code/pool_judge.py','--arm','qwen3-8b','--condition','C',
                    '--guided','--pairs','data/pool_pairs.csv',
                    '--shard',str(s),'--num-shards','4'], check=True)

## 4 · Merge and check

The merged file is the auxiliary variable. The two numbers that decide whether it is usable are the strict parse rate and the count of non-binding completions that produced a label — only the latter can contaminate a rate.

In [ ]:
import glob, json, collections
rows, man = [], []
for f in sorted(glob.glob('labels/pool_labels_qwen3-8b_C_guided_shard*.jsonl')):
    rows += [json.loads(l) for l in open(f) if l.strip()]
for f in sorted(glob.glob('labels/pool_manifest_*shard*.json')):
    man.append(json.load(open(f)))
keys = {(r['topic'], r['pmid']) for r in rows}
lab = [r for r in rows if r['label'] is not None]
print(f'rows {len(rows):,} | unique pairs {len(keys):,} | expected 117,562')
print(f'strict parse {len(lab)/max(len(rows),1):.4f} | nulls {len(rows)-len(lab):,}')
print(f'non-binding {sum(m["n_nonbinding"] for m in man):,} '
      f'| of them labelled {sum(m["n_nonbinding_labelled"] for m in man):,}')
print('eligible rate', sum(r['eligible'] or 0 for r in lab)/max(len(lab),1))
with open('pool_labels_qwen3-8b_C_guided.jsonl','w') as fh:
    for r in rows: fh.write(json.dumps(r)+'\n')
print('merged -> pool_labels_qwen3-8b_C_guided.jsonl')

## 5 · Download

Download the merged labels and the manifests. They are the inputs to the board stage, which needs no GPU.

In [ ]:
from google.colab import files
files.download('pool_labels_qwen3-8b_C_guided.jsonl')
import shutil, glob
shutil.make_archive('pool_manifests','zip','labels')
files.download('pool_manifests.zip')